# 06_GSE319709_qc_selection

**Thesis Methods section(s): 4.1.1, 4.1.3, 4.1.2**

**Reads:** GSE319709 CD45+ immune-cell matrices and metadata (GEO).

**Writes:** GSE319709_CD8_STRICT_postQC_scvi_rawcounts.h5ad

**Notes:** Order differs from the other datasets: CD8 selection runs BEFORE quality control. The gate is (CD3D > 0 or TRAC > 0) then (CD8A > 0 or CD8B > 0), with no CD4-exclusion step.

Input data are not included in this repository. Set `DATA_ROOT` below to a local folder
holding the GEO downloads; see `README.md` for accessions and the expected layout.


In [ ]:
# Root folder for input data (NOT included in this repository).
# Set the DATA_ROOT environment variable, or edit the fallback below.
import os
DATA_ROOT = os.environ.get("DATA_ROOT", "data")


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os, glob
import matplotlib.pyplot as plt


# -----------------------------
# Settings
# -----------------------------
DATA_DIR = f"{DATA_ROOT}/scVI/12 GSE319709"   # folder with GSM*.gz files
OUT_DIR  = f"{DATA_ROOT}/scVI/12 GSE319709"   # where to save outputs (change if you want)
os.makedirs(OUT_DIR, exist_ok=True)


# -----------------------------
# Helper: ordering sample IDs
# -----------------------------
def sample_sort_key(x):
    x = str(x)
    if x.startswith("PT-"):  return (0, int(x.split("-")[1]))
    if x.startswith("T-"):   return (1, int(x.split("-")[1]))
    if x.startswith("con-"): return (2, int(x.split("-")[1]))
    return (9, 9999)


# -----------------------------
# Helper: sparse-safe gene positive
# -----------------------------
def gene_pos(a, gene):
    if gene not in a.var_names:
        raise ValueError(f"Gene not found in var_names: {gene}")
    return (a[:, gene].X.toarray().ravel() > 0)


# ============================================================
# 1) Load full immune landscape (CD45+)
# ============================================================
mtx_files = sorted(glob.glob(os.path.join(DATA_DIR, "*-matrix.mtx.gz")))
print("Found mtx:", len(mtx_files))

adatas = []

for mtx in mtx_files:
    base = mtx.replace("-matrix.mtx.gz", "")
    feat = base + "-features.tsv.gz"
    bar  = base + "-barcodes.tsv.gz"

    # Load count matrix (genes x cells) -> transpose to cells x genes
    ad = sc.read_mtx(mtx).T

    # Features (1 col)
    genes = pd.read_csv(feat, header=None, sep="\t")
    ad.var["gene"] = genes.iloc[:, 0].astype(str).values
    ad.var_names = ad.var["gene"]
    ad.var_names_make_unique()

    # Barcodes
    bc = pd.read_csv(bar, header=None)
    ad.obs_names = bc.iloc[:, 0].astype(str).values

    # Sample info from filename: GSMxxxx_PT-1
    sample_name = os.path.basename(base)
    gsm, sample = sample_name.split("_", 1)

    # Make cell IDs unique across samples
    ad.obs_names = [f"{sample}_{x}" for x in ad.obs_names]

    # Metadata
    ad.obs["gsm_id"] = gsm
    ad.obs["sample_id"] = sample

    if sample.startswith("T-"):
        comp = "Tumour"
    elif sample.startswith("PT-"):
        comp = "Peritumour"
    elif sample.startswith("con-"):
        comp = "Blood"
    else:
        comp = "Unknown"

    ad.obs["compartment"] = comp
    ad.obs["dataset"] = "GSE319709"

    adatas.append(ad)

print("Loaded samples:", len(adatas))

adata = sc.concat(
    adatas,
    join="outer",
    label="batch",
    keys=[a.obs["sample_id"][0] for a in adatas],
    index_unique=None
)

print(adata)
print("\nCells per compartment (immune):")
print(adata.obs["compartment"].value_counts())

# ordered sample list
sample_order = sorted(adata.obs["sample_id"].unique(), key=sample_sort_key)

Pre-QC inspection plots — FULL IMMUNE landscape

In [ ]:
# Plot A: cells per sample_id (immune)
counts = adata.obs["sample_id"].value_counts().reindex(sample_order)

plt.figure(figsize=(12, 4))
plt.bar(counts.index.astype(str), counts.values)
plt.xticks(rotation=60, ha="right")
plt.ylabel("Number of cells")
plt.title("Immune landscape: cells per sample_id (pre-QC)")
plt.tight_layout()
plt.show()

# Plot B: stacked bar sample_id × compartment (sanity check)
tab = pd.crosstab(adata.obs["sample_id"], adata.obs["compartment"]).reindex(sample_order)
tab = tab[[c for c in ["Tumour","Peritumour","Blood"] if c in tab.columns]]

plt.figure(figsize=(12, 4))
bottom = np.zeros(len(tab))
for col in tab.columns:
    plt.bar(tab.index.astype(str), tab[col].values, bottom=bottom, label=col)
    bottom += tab[col].values

plt.xticks(rotation=60, ha="right")
plt.ylabel("Number of cells")
plt.title("Immune landscape: composition per sample_id (stacked by compartment, pre-QC)")
plt.legend(title="compartment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Plot C: cells per compartment (immune)
counts_comp = adata.obs["compartment"].value_counts().reindex(["Tumour","Peritumour","Blood"])

plt.figure(figsize=(6, 4))
plt.bar(counts_comp.index.astype(str), counts_comp.values)
plt.ylabel("Number of cells")
plt.title("Immune landscape: cells per compartment (pre-QC)")
plt.tight_layout()
plt.show()

Subset CD8 only (CD3 -> CD8 gate)

In [ ]:
# CD3 gate
cd3 = gene_pos(adata, "CD3D") | gene_pos(adata, "TRAC")
adata_cd3 = adata[cd3].copy()

# CD8 gate within CD3
cd8 = gene_pos(adata_cd3, "CD8A") | gene_pos(adata_cd3, "CD8B")
adata_cd8 = adata_cd3[cd8].copy()

print("\nSubsetting summary:")
print("All immune cells:", adata.n_obs)
print("CD3+ cells      :", adata_cd3.n_obs)
print("CD8+ (within CD3):", adata_cd8.n_obs)

print("\nCD8 cells per compartment:")
print(adata_cd8.obs["compartment"].value_counts())

sample_order_cd8 = sorted(adata_cd8.obs["sample_id"].unique(), key=sample_sort_key)

Pre-QC inspection plots — CD8-only object

In [ ]:
# Plot D: CD8 cells per sample_id
counts_cd8 = adata_cd8.obs["sample_id"].value_counts().reindex(sample_order_cd8)

plt.figure(figsize=(12, 4))
plt.bar(counts_cd8.index.astype(str), counts_cd8.values)
plt.xticks(rotation=60, ha="right")
plt.ylabel("Number of CD8 cells")
plt.title("CD8-only: cells per sample_id (pre-QC)")
plt.tight_layout()
plt.show()

# Plot E: stacked bar sample_id × compartment (should be single-compartment per sample)
tab_cd8 = pd.crosstab(adata_cd8.obs["sample_id"], adata_cd8.obs["compartment"]).reindex(sample_order_cd8)
tab_cd8 = tab_cd8[[c for c in ["Tumour","Peritumour","Blood"] if c in tab_cd8.columns]]

plt.figure(figsize=(12, 4))
bottom = np.zeros(len(tab_cd8))
for col in tab_cd8.columns:
    plt.bar(tab_cd8.index.astype(str), tab_cd8[col].values, bottom=bottom, label=col)
    bottom += tab_cd8[col].values

plt.xticks(rotation=60, ha="right")
plt.ylabel("Number of CD8 cells")
plt.title("CD8-only: composition per sample_id (stacked by compartment, pre-QC)")
plt.legend(title="compartment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Plot F: CD8 cells per compartment
counts_comp_cd8 = adata_cd8.obs["compartment"].value_counts().reindex(["Tumour","Peritumour","Blood"])

plt.figure(figsize=(6, 4))
plt.bar(counts_comp_cd8.index.astype(str), counts_comp_cd8.values)
plt.ylabel("Number of CD8 cells")
plt.title("CD8-only: cells per compartment (pre-QC)")
plt.tight_layout()
plt.show()

In [ ]:
import os
os.getcwd()

In [ ]:
import os

os.chdir(f"{DATA_ROOT}/scVI/12 GSE319709")
os.getcwd()

In [ ]:
import re

def add_patient_id(adata_obj):
    # extract trailing integer from sample_id (e.g., "PT-3" -> 3, "con-8" -> 8)
    nums = adata_obj.obs["sample_id"].astype(str).str.extract(r"-(\d+)$")[0].astype(int)
    adata_obj.obs["patient_num"] = nums
    adata_obj.obs["patient_id"] = nums.map(lambda x: f"P{int(x):02d}")
    return adata_obj

adata_cd8 = add_patient_id(adata)
# if you already created CD8 object:
# adata_cd8 = add_patient_id(adata_cd8)

adata_cd8.obs[["sample_id","patient_id","compartment"]].drop_duplicates().sort_values(["patient_id","sample_id"])

In [ ]:
print("Unique patients:", adata_cd8.obs["patient_id"].nunique())
print("Patients:", sorted(adata_cd8.obs["patient_id"].unique()))
print("\nCompartment coverage per patient:")
print(pd.crosstab(adata_cd8.obs["patient_id"], adata_cd8.obs["compartment"]))

Patient-level plots (CD8)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use post-QC CD8 object
a = adata_cd8.copy()   # change to adata_cd8 if needed

# Ensure patient_id exists
assert "patient_id" in a.obs.columns

# Order patients numerically (P01, P02, ...)
patient_order = sorted(a.obs["patient_id"].unique())

# -----------------------------
# 1) Total CD8 cells per patient
# -----------------------------
counts_patient = a.obs["patient_id"].value_counts().reindex(patient_order)

plt.figure(figsize=(8,4))
plt.bar(counts_patient.index, counts_patient.values)
plt.ylabel("CD8 cells")
plt.title("CD8 cells per patient (post-QC)")
plt.tight_layout()
plt.show()


# -----------------------------
# 2) Stacked bar: Patient × Compartment
# -----------------------------
tab = pd.crosstab(a.obs["patient_id"], a.obs["compartment"]).reindex(patient_order)
tab = tab[[c for c in ["Tumour","Peritumour","Blood"] if c in tab.columns]]

plt.figure(figsize=(8,4))
bottom = np.zeros(len(tab))
for col in tab.columns:
    plt.bar(tab.index, tab[col].values, bottom=bottom, label=col)
    bottom += tab[col].values

plt.ylabel("CD8 cells")
plt.title("CD8 composition per patient (stacked)")
plt.legend(title="Compartment", bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout()
plt.show()


# -----------------------------
# 3) Fractional composition (very important plot)
# -----------------------------
frac = tab.div(tab.sum(axis=1), axis=0)

plt.figure(figsize=(8,4))
bottom = np.zeros(len(frac))
for col in frac.columns:
    plt.bar(frac.index, frac[col].values, bottom=bottom, label=col)
    bottom += frac[col].values

plt.ylabel("Fraction of CD8 cells")
plt.title("CD8 compartment fractions per patient")
plt.legend(title="Compartment", bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout()
plt.show()

Save CD8-only PRE-QC rawcounts

In [ ]:
cd8_preqc_fp = os.path.join(OUT_DIR, "GSE319709_CD8_preQC_rawcounts.h5ad")
adata_cd8.write(cd8_preqc_fp)

print("\n✅ Saved CD8-only pre-QC object:")
print(cd8_preqc_fp)
print(adata_cd8)

GSE319709 — CD8-only QC (from adata_cd8 PRE-QC)
QC metrics + plots → filtering → save postQC scVI rawcounts

In [ ]:
import scanpy as sc
import numpy as np
import os

# -----------------------------
# Settings
# -----------------------------
OUT_DIR = f"{DATA_ROOT}/scVI/12 GSE319709"
os.makedirs(OUT_DIR, exist_ok=True)

QC metrics

In [ ]:
adata_cd8.var["mt"] = adata_cd8.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata_cd8, qc_vars=["mt"], inplace=True)

print(adata_cd8)
print("\nQC summary:")
print(adata_cd8.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe())

QC plots (overall + by compartment + by sample)

In [ ]:
sc.pl.violin(
    adata_cd8,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.2,
    multi_panel=True
)

sc.pl.violin(
    adata_cd8,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    groupby="compartment",
    jitter=0.2,
    multi_panel=True
)

sc.pl.scatter(adata_cd8, x="total_counts", y="pct_counts_mt", color="compartment")
sc.pl.scatter(adata_cd8, x="total_counts", y="n_genes_by_counts", color="compartment")

# optional (18 samples → OK)
sc.pl.violin(
    adata_cd8,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    groupby="sample_id",
    jitter=0.15,
    rotation=90,
    multi_panel=True
)

Filtering thresholds

In [ ]:
min_genes  = 200
min_counts = 500
max_pct_mt = 15

# robust upper bounds to remove extreme outliers
max_genes  = float(np.quantile(adata_cd8.obs["n_genes_by_counts"], 0.995))
max_counts = float(np.quantile(adata_cd8.obs["total_counts"], 0.995))

print("\nQC thresholds:")
print({
    "min_genes": min_genes,
    "min_counts": min_counts,
    "max_pct_mt": max_pct_mt,
    "max_genes_q0.995": max_genes,
    "max_counts_q0.995": max_counts
})

Apply filtering

In [ ]:
before = adata_cd8.n_obs

adata_cd8_qc = adata_cd8[
    (adata_cd8.obs["n_genes_by_counts"] >= min_genes) &
    (adata_cd8.obs["total_counts"]      >= min_counts) &
    (adata_cd8.obs["pct_counts_mt"]     <= max_pct_mt) &
    (adata_cd8.obs["n_genes_by_counts"] <= max_genes) &
    (adata_cd8.obs["total_counts"]      <= max_counts)
].copy()

after = adata_cd8_qc.n_obs

print("\nCells before:", before)
print("Cells after :", after)
print("Fraction kept:", after / before)

print("\nPost-QC cells by compartment:")
print(adata_cd8_qc.obs["compartment"].value_counts())

print("\nPost-QC cells by sample_id:")
print(adata_cd8_qc.obs["sample_id"].value_counts().sort_index())

Post-QC sanity plots (quick)

In [ ]:
sc.pl.violin(
    adata_cd8_qc,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    groupby="compartment",
    jitter=0.2,
    multi_panel=True
)

sc.pl.scatter(adata_cd8_qc, x="total_counts", y="pct_counts_mt", color="compartment")


Save post-QC scVI-ready rawcounts

In [ ]:
postqc_fp = os.path.join(OUT_DIR, "GSE319709_CD8_postQC_scvi_rawcounts.h5ad")
adata_cd8_qc.write(postqc_fp)

print("\n✅ Saved CD8 post-QC scVI-ready object:")
print(postqc_fp)
print(adata_cd8_qc)